# FrameGuard: full training run on a Kaggle GPU

Trains FrameGuard from scratch (EfficientNet-B0 spatial CNN, then LSTM) and evaluates it on the
FaceForensics++ internal test split and the Celeb-DF v2 external set.

## Before you run

1. **Add data** (right panel → *Add Input*), both of these datasets:
   - `xdxd003/ff-c23` (FaceForensics++ C23)
   - `reubensuju/celeb-df-v2` (Celeb-DF v2)
2. **Settings → Accelerator → GPU** (T4 or P100).
3. **Settings → Internet → On** (the notebook clones the code and installs MediaPipe).
   Kaggle requires a phone-verified account for GPU and internet.
4. To run in the background with the browser closed: **Save Version → Save & Run All (Commit)**.
   Results are kept in the version's **Output** tab. A run has a 12-hour limit.

The run first rebuilds the train/validation/test split and checks it against
`splits/reference-split.csv` from the original run, so results stay comparable.

In [ ]:
# Settings. Defaults are the fast GPU configuration.
EPOCHS = 5              # per training stage (spatial CNN and LSTM)
BATCH_SIZE = 32
FRAMES_PER_VIDEO = 32   # spatial crops drawn per training video per epoch; None = every crop (~4x slower)
WORKERS = 4             # DataLoader worker processes
MAX_SAMPLES = None      # frames sampled per video during face preparation; None = 256 if disk allows
REPO = "https://github.com/khakash-060206/FrameGuard.git"

In [ ]:
import os, shutil, subprocess, sys
print(sys.version)
subprocess.run(["nvidia-smi"], check=False)
for path in ("/kaggle/working", "/kaggle/temp", "/tmp"):
    if os.path.exists(path):
        print(f"{path}: {shutil.disk_usage(path).free / 1024**3:.0f} GB free")

## 1. Get the code and install dependencies

In [ ]:
PROJECT = "/kaggle/working/FrameGuard"
if not os.path.exists(PROJECT):
    subprocess.run(["git", "clone", "--depth", "1", REPO, PROJECT], check=True)
else:
    subprocess.run(["git", "-C", PROJECT, "pull", "--ff-only"], check=True)
os.chdir(PROJECT)
subprocess.run(["git", "log", "--oneline", "-1"], check=True)

In [ ]:
# Kaggle already has PyTorch, torchvision, NumPy and scikit-learn. MediaPipe is the only missing package.
# Remove Kaggle's plain OpenCV builds first so only one cv2 (opencv-contrib, pulled in by MediaPipe) remains.
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "opencv-python", "opencv-python-headless"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "mediapipe==0.10.35"], check=True)

In [ ]:
# Verify the stack in a fresh process (the kernel may still hold the old cv2).
check = '''
import torch, torchvision, cv2, mediapipe, numpy, sklearn
print("torch", torch.__version__, "| torchvision", torchvision.__version__, "| opencv", cv2.__version__,
      "| mediapipe", mediapipe.__version__, "| numpy", numpy.__version__, "| sklearn", sklearn.__version__)
assert torch.cuda.is_available(), "No GPU: set Settings -> Accelerator -> GPU"
print("GPU:", torch.cuda.get_device_name(0))
'''
subprocess.run([sys.executable, "-c", check], check=True)

## 2. Find the datasets and choose where to store face crops

In [ ]:
from pathlib import Path

def find(marker, kind):
    for path in sorted(Path("/kaggle/input").rglob(marker)):
        if (kind == "dir" and path.is_dir()) or (kind == "file" and path.is_file()):
            return path.parent
    raise FileNotFoundError(f"{marker} not found under /kaggle/input. Add the dataset (see the top of the notebook).")

FFPP = find("FaceForensics++_C23", "dir")
CELEB = find("List_of_testing_videos.txt", "file")
print("FF++ :", FFPP)
print("Celeb:", CELEB)

# Face crops are ~42 GB at 256 frames/video, so they must not go in /kaggle/working (20 GB output limit).
candidates = [p for p in ("/kaggle/temp", "/tmp") if os.path.isdir(p)]
DATA_DIR = max(candidates, key=lambda p: shutil.disk_usage(p).free) + "/frameguard-data"
free_gb = shutil.disk_usage(os.path.dirname(DATA_DIR)).free / 1024**3
if MAX_SAMPLES is None:
    # ~0.165 GB of crops per sampled frame position across the dataset, plus a safety margin.
    MAX_SAMPLES = next((m for m in (256, 128, 64) if 0.165 * m + 8 < free_gb), None)
    if MAX_SAMPLES is None:
        raise RuntimeError(f"Only {free_gb:.0f} GB free; at least ~19 GB is needed for face crops.")
print(f"Data dir: {DATA_DIR} ({free_gb:.0f} GB free) | max frames per video: {MAX_SAMPLES}")
if MAX_SAMPLES != 256:
    print("NOTE: fewer frames per video than the original 256-frame protocol; report this with the results.")

## 3. Run the full experiment

Import → split check → face crops → spatial CNN → feature cache → LSTM → evaluation. Progress streams below and is also written to `runs/experiment-status.json`.

In [ ]:
command = [sys.executable, "-u", "scripts/run_experiment.py",
           "--ffpp", str(FFPP), "--celeb", str(CELEB), "--data-dir", DATA_DIR,
           "--device", "cuda", "--epochs", str(EPOCHS), "--batch-size", str(BATCH_SIZE),
           "--max-samples", str(MAX_SAMPLES), "--workers", str(WORKERS), "--amp", "on", "--resume"]
if FRAMES_PER_VIDEO:
    command += ["--frames-per-video", str(FRAMES_PER_VIDEO)]
print(" ".join(command), flush=True)

with open("runs-console.log", "w") as log:
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        log.write(line)
        # MediaPipe prints harmless telemetry upload errors; keep them out of the notebook view.
        if "clearcut" not in line and "Source Location Trace" not in line and "portable_clearcut" not in line:
            print(line, end="", flush=True)
    code_ = process.wait()
if code_ != 0:
    raise RuntimeError(f"Experiment failed (exit code {code_}). See runs/experiment-status.json and runs-console.log.")

## 4. Results

In [ ]:
import json

def show(path):
    report = json.loads(Path(path).read_text())
    print(f"\n=== {path} (split: {report['split']}, unit: {report['unit']}) ===")
    print(f"{'model':<10}{'dataset':<18}{'videos':>7}{'ROC-AUC':>9}{'F1':>7}{'Prec':>7}{'Recall':>8}{'FPR':>7}")
    for model, domains in report["metrics"].items():
        for domain, m in domains.items():
            print(f"{model:<10}{domain:<18}{m['count']:>7}{m['roc_auc']:>9.3f}{m['f1']:>7.3f}"
                  f"{m['precision']:>7.3f}{m['recall']:>8.3f}{m['false_positive_rate']:>7.3f}")
    print(f"Unassessed videos: {len(report['unassessed'])}")

for stage in ("spatial", "temporal"):
    history = json.loads(Path(f"models/{stage}.history.json").read_text())
    best = max(history, key=lambda e: e["validation"]["roc_auc"])
    print(f"{stage}: best validation ROC-AUC {best['validation']['roc_auc']:.3f} at epoch {best['epoch']}/{len(history)}")
show("runs/ffpp-test.json")
show("runs/celeb-external.json")

In [ ]:
# Bundle everything needed to view the results locally into one downloadable file (Output tab).
import zipfile
bundle = Path("/kaggle/working/frameguard-results.zip")
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as archive:
    for pattern in ("models/*.pt", "models/*.history.json", "runs/*.json", "runs-console.log"):
        for file in Path(".").glob(pattern):
            archive.write(file, file.as_posix())
print(bundle, f"{bundle.stat().st_size / 1024**2:.1f} MB")

## Viewing results on your own computer

Download `frameguard-results.zip` from the version's **Output** tab, unzip it into your local
FrameGuard folder (it contains `models/` and `runs/`), then run:

```
.\.venv\Scripts\python.exe -m frameguard dashboard
```

and open http://127.0.0.1:8765.